In [2]:
import os
print(os.getcwd())  # shows which folder Jupyter is currently in

C:\Users\hp\Downloads\Movie Recommendation


In [3]:
import os
print(os.path.exists(".env"))  # should print True

False


In [2]:
import os
os.environ["TMDB_API_KEY"] = "7e428471a814d5d6c0a5016997fd0ef0"
API_KEY = os.environ.get("TMDB_API_KEY")
BASE = "https://api.themoviedb.org/3"

print("Key loaded:", API_KEY[:6], "...")

Key loaded: 7e4284 ...


In [12]:
import requests
import pandas as pd
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Create a session with automatic retries
session = requests.Session()
retry = Retry(connect=3, backoff_factor=1)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)

def fetch_movies(category, pages=5):
    results = []
    for page in range(1, pages + 1):
        try:
            r = session.get(f"{BASE}/movie/{category}",
                            params={"api_key": API_KEY, "page": page},
                            timeout=10)
            data = r.json().get("results", [])
            for movie in data:
                movie["category"] = category
            results.extend(data)
            time.sleep(0.5)  # slightly longer delay
        except Exception as e:
            print(f"  Skipping page {page} due to error: {e}")
            time.sleep(2)   # wait longer before continuing
            continue
    print(f"{category}: {len(results)} movies fetched")
    return results

old     = fetch_movies("top_rated", pages=20)
current = fetch_movies("now_playing", pages=5)
future  = fetch_movies("upcoming", pages=5)

df = pd.DataFrame(old + current + future)
df = df.drop_duplicates(subset="id").reset_index(drop=True)
print(f"\nTotal unique movies: {df.shape[0]}")
df.head()

top_rated: 400 movies fetched
now_playing: 100 movies fetched
upcoming: 100 movies fetched

Total unique movies: 565


,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count,category
0,False,/zMwhWailP1WY7sb6AoE6b8ugoy.jpg,"[12, 16, 10751, 14]",1007757,Swapped,en,Swapped,"A small woodland creature and a majestic bird,...",229.3482,/tHhxWxge06goXU6ZQH1hj7vK8Hd.jpg,2026-05-01,False,False,8.974,1618,top_rated
1,False,/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg,"[18, 80]",278,The Shawshank Redemption,en,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,82.1095,/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,1994-09-23,False,False,8.722,30513,top_rated
2,False,/tSPT36ZKlP2WVHJLM4cQPLSzv3b.jpg,"[18, 80]",238,The Godfather,en,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",44.5504,/3bhkrj58Vtu7enYsRolD1fZdja1.jpg,1972-03-14,False,False,8.687,23004,top_rated
3,False,/2I1OFQJ0L9T0dpU6FobKFWV2PxX.jpg,"[878, 12]",687163,Project Hail Mary,en,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,251.2943,/yihdXomYb5kTeSivtFndMy5iDmf.jpg,2026-03-15,False,False,8.666,4775,top_rated
4,False,/kGzFbGhp99zva6oZODW5atUtnqi.jpg,"[18, 80]",240,The Godfather Part II,en,The Godfather Part II,In the continuing saga of the Corleone crime f...,28.5329,/hek3koDUyRQk7FIhPXsa6mT2Zc3.jpg,1974-12-20,False,False,8.572,13948,top_rated


In [6]:
pip install jupyter-ai

Defaulting to user installation because normal site-packages is not writeable
INFO: pip is looking at multiple versions of jupyterlab-chat to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of jupyterlab-chat to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of jupyter-server-documents to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of jupyter-collaboration to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of jupyter-server-ydoc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of pycrdt-websocket to determine which version is compatible with other requirements. This coul

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [10]:
r = requests.get(f"{BASE}/movie/top_rated",
                 params={"api_key": API_KEY, "page": 1})

print("Status code:", r.status_code)
print("Response:", r.text[:300])  # print first 300 characters

Status code: 200
Response: {"page":1,"results":[{"adult":false,"backdrop_path":"/zMwhWailP1WY7sb6AoE6b8ugoy.jpg","genre_ids":[12,16,10751,14],"id":1007757,"title":"Swapped","original_language":"en","original_title":"Swapped","overview":"A small woodland creature and a majestic bird, two natural sworn enemies of the Valley, ma


In [13]:
def get_details(movie_id):
    try:
        r = session.get(f"{BASE}/movie/{movie_id}",
                        params={"api_key": API_KEY,
                                "append_to_response": "credits,keywords"},
                        timeout=10)
        d = r.json()

        genres   = " ".join([g["name"] for g in d.get("genres", [])])
        cast     = " ".join([c["name"] for c in d.get("credits", {})
                             .get("cast", [])[:3]])
        director = next((c["name"] for c in d.get("credits", {})
                         .get("crew", []) if c["job"] == "Director"), "")
        keywords = " ".join([k["name"] for k in d.get("keywords", {})
                             .get("keywords", [])])

        return {"genres": genres, "cast": cast,
                "director": director, "keywords": keywords}
    except:
        return {"genres": "", "cast": "", "director": "", "keywords": ""}

print("Fetching details for each movie...")
details = []
for i, mid in enumerate(df["id"]):
    details.append(get_details(mid))
    time.sleep(0.25)
    if i % 50 == 0:
        print(f"  {i}/{len(df)} done...")

details_df = pd.DataFrame(details)
df = pd.concat([df.reset_index(drop=True), details_df], axis=1)
print("\nDone! Columns now:", list(df.columns))

Fetching details for each movie...
  0/565 done...
  50/565 done...
  100/565 done...
  150/565 done...
  200/565 done...
  250/565 done...
  300/565 done...
  350/565 done...
  400/565 done...
  450/565 done...
  500/565 done...
  550/565 done...

Done! Columns now: ['adult', 'backdrop_path', 'genre_ids', 'id', 'title', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'release_date', 'softcore', 'video', 'vote_average', 'vote_count', 'category', 'genres', 'cast', 'director', 'keywords']


In [14]:
# Fill missing values
df[["genres", "cast", "director", "keywords", "overview"]] = \
    df[["genres", "cast", "director", "keywords", "overview"]].fillna("")

# Drop movies with no title
df.dropna(subset=["title"], inplace=True)

# Build poster URL
df["poster_url"] = "https://image.tmdb.org/t/p/w500" + \
                    df["poster_path"].fillna("")

# Extract year from release date
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["year"] = df["release_date"].dt.year

# THE MOST IMPORTANT COLUMN — ML model uses this
df["soup"] = (df["genres"] + " " +
              df["cast"] + " " +
              df["director"] + " " +
              df["keywords"])

# Keep only needed columns
keep = ["id", "title", "overview", "soup", "genres", "cast", "director",
        "vote_average", "vote_count", "release_date", "year",
        "poster_url", "category"]
df = df[[c for c in keep if c in df.columns]]

print("Shape:", df.shape)
print("\nSample soup value:")
print(df["soup"].iloc[0])
print("\nCleaning done!")

Shape: (565, 13)

Sample soup value:
Adventure Animation Family Fantasy Michael B. Jordan Juno Temple Tracy Morgan Nathan Greno wolf buddy forest fire woodlands forest lore bird 3d animation familiar vibrant body swap animal adventure

Cleaning done!


In [15]:
df.to_csv("movies.csv", index=False)
print("Saved! movies.csv is ready with", len(df), "movies")

Saved! movies.csv is ready with 565 movies


In [17]:
import pandas as pd

df = pd.read_csv("movies.csv")
print("Loaded:", df.shape)
df.head()

Loaded: (565, 13)


,id,title,overview,soup,genres,cast,director,vote_average,vote_count,release_date,year,poster_url,category
0,1007757,Swapped,"A small woodland creature and a majestic bird,...",Adventure Animation Family Fantasy Michael B. ...,Adventure Animation Family Fantasy,Michael B. Jordan Juno Temple Tracy Morgan,Nathan Greno,8.974,1618,2026-05-01,2026,https://image.tmdb.org/t/p/w500/tHhxWxge06goXU...,top_rated
1,278,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,Drama Crime Tim Robbins Morgan Freeman Bob Gun...,Drama Crime,Tim Robbins Morgan Freeman Bob Gunton,Frank Darabont,8.722,30513,1994-09-23,1994,https://image.tmdb.org/t/p/w500/9cqNxx0GxF0bfl...,top_rated
2,238,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",Drama Crime Marlon Brando Al Pacino James Caan...,Drama Crime,Marlon Brando Al Pacino James Caan,Francis Ford Coppola,8.687,23004,1972-03-14,1972,https://image.tmdb.org/t/p/w500/3bhkrj58Vtu7en...,top_rated
3,687163,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,Science Fiction Adventure Ryan Gosling Sandra ...,Science Fiction Adventure,Ryan Gosling Sandra Hüller James Ortiz,Phil Lord,8.666,4775,2026-03-15,2026,https://image.tmdb.org/t/p/w500/yihdXomYb5kTeS...,top_rated
4,240,The Godfather Part II,In the continuing saga of the Corleone crime f...,Drama Crime Al Pacino Robert Duvall Diane Keat...,Drama Crime,Al Pacino Robert Duvall Diane Keaton,Francis Ford Coppola,8.572,13948,1974-12-20,1974,https://image.tmdb.org/t/p/w500/hek3koDUyRQk7F...,top_rated


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Fill any remaining nulls in soup
df["soup"] = df["soup"].fillna("")

# Step 1 — Convert soup text into numbers (TF-IDF matrix)
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["soup"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (565, 5997)


In [19]:
# Step 2 — Compute cosine similarity between all movies
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Cosine similarity matrix shape:", cosine_sim.shape)
print("Sample similarity scores for movie 0:")
print(cosine_sim[0][:10])  # first 10 scores of first movie

Cosine similarity matrix shape: (565, 565)
Sample similarity scores for movie 0:
[1.         0.03205617 0.00641623 0.01514312 0.         0.
 0.         0.         0.03861581 0.        ]


In [20]:
# Map movie title to its index for fast lookup
indices = pd.Series(df.index, index=df["title"]).drop_duplicates()

def recommend(title, n=10):
    # Check if movie exists
    if title not in indices:
        print(f"'{title}' not found in dataset.")
        return
    
    idx = indices[title]
    
    # Get similarity scores for this movie vs all others
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort by similarity, highest first, skip the movie itself
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:n+1]
    
    # Get movie indices
    movie_indices = [i[0] for i in sim_scores]
    
    # Return results
    result = df.iloc[movie_indices][["title", "genres", "vote_average", "year"]].copy()
    result["similarity"] = [round(i[1], 3) for i in sim_scores]
    return result

# Test it!
print("=== Recommendations for Inception ===")
print(recommend("Inception"))

=== Recommendations for Inception ===
                                     title  \
201  Eternal Sunshine of the Spotless Mind   
519                              Affection   
269                       Ultraman: Rising   
133                                Memento   
405                            In the Grey   
3                        Project Hail Mary   
89                              The Matrix   
564        The Girl Who Leapt Through Time   
448                              Protector   
318                    Catch Me If You Can   

                                      genres  vote_average  year  similarity  
201            Science Fiction Drama Romance         8.091  2004       0.127  
519                   Horror Science Fiction         6.286  2026       0.122  
269  Animation Science Fiction Family Action         8.028  2024       0.117  
133                         Mystery Thriller         8.200  2000       0.107  
405                          Action Thriller         6.600  

In [21]:
import pickle

with open("recommender_model.pkl", "wb") as f:
    pickle.dump({
        "cosine_sim": cosine_sim,
        "indices": indices,
        "df": df
    }, f)

print("Model saved as recommender_model.pkl")
print("Phase 2 complete! 🎉")

Model saved as recommender_model.pkl
Phase 2 complete! 🎉


In [22]:
import pickle

with open("recommender_model.pkl", "wb") as f:
    pickle.dump({
        "cosine_sim": cosine_sim,
        "indices": indices,
        "df": df
    }, f)

print("Model saved as recommender_model.pkl")
print("Phase 2 complete!")

Model saved as recommender_model.pkl
Phase 2 complete!


In [3]:
import pickle
with open("recommender_model.pkl", "rb") as f:
    model = pickle.load(f)

df = model["df"]
print(df.columns.tolist())
print(df.head(2))
